### Requirements
This notebook expects a Python environment with:
- numpy, pandas
- scikit-learn
- catboost

And a data file located at:
`data/btc_yahoo_2019_2021.csv`


## 1) Setup: imports + file path

This notebook builds a *market-only* machine learning dataset from Bitcoin OHLCV data (Yahoo Finance),
then trains baseline models and stronger models to predict:

- next-day volatility (regression)
- next-day direction (classification)
- next-day high-volatility regimes (classification)

First we just import the basic libraries and set the path to the data file.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/btc_yahoo_2019_2021.csv"

## 2) Load the Yahoo Finance CSV (and clean the column format)

Some exports from Yahoo Finance / yfinance create “double headers” (like a MultiIndex),
so we load using a 2-row header and then flatten it if needed.

We also:
- remove accidental “Unnamed” index columns
- parse the Date column
- sort the dataset chronologically

This ensures the time series is clean and in the right order.

In [2]:
df = pd.read_csv(DATA_PATH, header=[0,1])

# Flatten columns if it's a MultiIndex from yfinance export
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] for c in df.columns]

# Drop unnamed index columns (common after CSV export)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print(df.head(3))
print(df.columns)
print(df["Date"].min(), df["Date"].max())

        Date          Open          High          Low         Close  \
0 2018-01-31  10108.200195  10381.599609  9777.419922  10221.099609   
1 2018-02-01  10237.299805  10288.799805  8812.280273   9170.540039   
2 2018-02-02   9142.280273   9142.280273  7796.490234   8830.750000   

        Volume  log_return     vol_7    vol_14    vol_30  log_volume  \
0   8041160192    0.011295  0.048859  0.058188  0.066341   22.807839   
1   9959400448   -0.108458  0.059980  0.063328  0.065809   23.021783   
2  12726899712   -0.037756  0.059256  0.063051  0.065655   23.266984   

   vol_change  direction_t1  vol_14_t1  
0   -0.071581             0   0.063328  
1    0.213943             0   0.063051  
2    0.245201             1   0.054811  
Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'log_return', 'vol_7',
       'vol_14', 'vol_30', 'log_volume', 'vol_change', 'direction_t1',
       'vol_14_t1'],
      dtype='object')
2018-01-31 00:00:00 2021-12-30 00:00:00


## 3) Keep only raw OHLCV (clean starting point)

the CSV already contains engineered columns (returns, rolling vol, etc.),
we intentionally drop them and keep only:
Date, Open, High, Low, Close, Volume**


In [3]:
base_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]
missing = [c for c in base_cols if c not in df.columns]
print("Missing:", missing)

df = df[base_cols].copy()
df.head()

Missing: []


,Date,Open,High,Low,Close,Volume
0,2018-01-31,10108.200195,10381.599609,9777.419922,10221.099609,8041160192
1,2018-02-01,10237.299805,10288.799805,8812.280273,9170.540039,9959400448
2,2018-02-02,9142.280273,9142.280273,7796.490234,8830.750000,12726899712
3,2018-02-03,8852.120117,9430.750000,8251.629883,9174.910156,7263790080
4,2018-02-04,9175.700195,9334.870117,8031.220215,8277.009766,7073549824


In [4]:
# Keep only raw OHLCV columns (we rebuild features & targets ourselves)
df = df[["Date", "Open", "High", "Low", "Close", "Volume"]].copy()

df.head()


,Date,Open,High,Low,Close,Volume
0,2018-01-31,10108.200195,10381.599609,9777.419922,10221.099609,8041160192
1,2018-02-01,10237.299805,10288.799805,8812.280273,9170.540039,9959400448
2,2018-02-02,9142.280273,9142.280273,7796.490234,8830.750000,12726899712
3,2018-02-03,8852.120117,9430.750000,8251.629883,9174.910156,7263790080
4,2018-02-04,9175.700195,9334.870117,8031.220215,8277.009766,7073549824


## 4) Create market features (inputs)

Now we build features that are available at the end of day T.
These will be used to predict outcomes on day T+1.

Feature groups:
- Returns (1d, 3d, 7d, 14d, 30d) → momentum / trend
- High–Low log range → intraday uncertainty
- Rolling volatility of returns → volatility clustering
- Volume (log + z-scores) → unusual activity

These are standard “market-only” signals used in many finance ML baselines.


In [ ]:
# Feature engineering (past-only features)

# Log returns
df["ret_1d"] = np.log(df["Close"]).diff()

# Multi-day returns
for w in [3, 7, 14, 30]:
    df[f"ret_{w}d"] = np.log(df["Close"]).diff(w)

# Intraday range (volatility proxy)
df["hl_range"] = np.log(df["High"] / df["Low"])

# Rolling volatility of returns
for w in [7, 14, 30]:
    df[f"rv_{w}d"] = df["ret_1d"].rolling(w).std()

# Volume features
df["log_volume"] = np.log(df["Volume"].replace(0, np.nan))
for w in [7, 30]:
    df[f"volz_{w}d"] = (
        df["log_volume"] - df["log_volume"].rolling(w).mean()
    ) / df["log_volume"].rolling(w).std()

df.head()


,Date,Open,High,Low,Close,Volume,ret_1d,ret_3d,ret_7d,ret_14d,ret_30d,hl_range,rv_7d,rv_14d,rv_30d,log_volume,volz_7d,volz_30d
0,2018-01-31,10108.200195,10381.599609,9777.419922,10221.099609,8041160192,NaN,NaN,NaN,NaN,NaN,0.059959,NaN,NaN,NaN,22.807839,NaN,NaN
1,2018-02-01,10237.299805,10288.799805,8812.280273,9170.540039,9959400448,-0.108458,NaN,NaN,NaN,NaN,0.154910,NaN,NaN,NaN,23.021783,NaN,NaN
2,2018-02-02,9142.280273,9142.280273,7796.490234,8830.750000,12726899712,-0.037756,NaN,NaN,NaN,NaN,0.159236,NaN,NaN,NaN,23.266984,NaN,NaN
3,2018-02-03,8852.120117,9430.750000,8251.629883,9174.910156,7263790080,0.038233,-0.107982,NaN,NaN,NaN,0.133565,NaN,NaN,NaN,22.706168,NaN,NaN
4,2018-02-04,9175.700195,9334.870117,8031.220215,8277.009766,7073549824,-0.102991,-0.102514,NaN,NaN,NaN,0.150420,NaN,NaN,NaN,22.679628,NaN,NaN


## 5) Define prediction targets (what we want to predict)

We create *next-day* targets so the prediction setup is realistic:

1. Next-day volatility (regression) : We use Parkinson volatility computed from High/Low.

2. Next-day direction (classification) : 1 if next-day return is positive, else 0.

3. Next-day high-volatility regime (classification): High-vol is defined as “top 20% volatility” using a rolling 90-day threshold.

Important: targets use .shift(-1) so the label truly belongs to “tomorrow”.


In [ ]:
#  Target construction (next-day targets)

# Parkinson volatility proxy (computed for day t)
df["parkinson_vol"] = np.sqrt(1 / (4 * np.log(2))) * np.abs(
    np.log(df["High"] / df["Low"])
)

# Target 1: next-day volatility (regression)
df["y_vol_next"] = df["parkinson_vol"].shift(-1)

# Target 2: next-day price direction (classification)
df["y_up"] = (df["ret_1d"].shift(-1) > 0).astype(int)

# Target 3: high-volatility regime (classification)
# Define threshold as rolling top 20% of historical volatility
roll = 90
vol_threshold = df["parkinson_vol"].rolling(roll).quantile(0.80)

df["y_high_vol"] = (df["y_vol_next"] > vol_threshold).astype(int)

df[[
    "Date",
    "parkinson_vol",
    "y_vol_next",
    "y_up",
    "y_high_vol"
]].head(10)


,Date,parkinson_vol,y_vol_next,y_up,y_high_vol
0,2018-01-31,0.036009,0.093033,0,0
1,2018-02-01,0.093033,0.095631,0,0
2,2018-02-02,0.095631,0.080214,1,0
3,2018-02-03,0.080214,0.090337,0,0
4,2018-02-04,0.090337,0.128223,0,0
5,2018-02-05,0.128223,0.156646,1,0
6,2018-02-06,0.156646,0.097267,0,0
7,2018-02-07,0.097267,0.068367,1,0
8,2018-02-08,0.068367,0.061641,1,0
9,2018-02-09,0.061641,0.057077,0,0


## 6) Assemble the final ML dataset (features + targets)

Here we select:
- the feature columns (X)
- the target columns (y)

We then drop rows with NaNs created by rolling windows and shifting.
This produces a clean table ready for modeling:

- each row corresponds to a day T
- features are known by the end of day T
- labels correspond to day T+1


In [ ]:
#  Final dataset assembly 

# Feature columns (everything known at end of day T)
feature_cols = [
    "ret_1d",
    "ret_3d",
    "ret_7d",
    "ret_14d",
    "ret_30d",
    "hl_range",
    "rv_7d",
    "rv_14d",
    "rv_30d",
    "log_volume",
    "volz_7d",
    "volz_30d",
    "parkinson_vol"
]

target_cols = [
    "y_vol_next",
    "y_up",
    "y_high_vol"
]

dataset = df[["Date"] + feature_cols + target_cols].dropna().reset_index(drop=True)

dataset.head(), dataset.shape



(        Date    ret_1d    ret_3d    ret_7d   ret_14d   ret_30d  hl_range  \
 0 2018-03-02  0.012288  0.033086  0.073469  0.080013  0.081265  0.030757   
 1 2018-03-03  0.035732  0.099847  0.157736  0.033362  0.225455  0.046683   
 2 2018-03-04  0.001991  0.050011  0.174959  0.087146  0.265202  0.033250   
 3 2018-03-05  0.005259  0.042982  0.110102  0.030531  0.232228  0.022482   
 4 2018-03-06 -0.071017 -0.063768  0.005050 -0.056255  0.264202  0.072645   
 
       rv_7d    rv_14d    rv_30d  log_volume   volz_7d  volz_30d  \
 0  0.044142  0.051861  0.069685   22.754120  1.002971 -0.220997   
 1  0.036122  0.047904  0.066660   22.623965 -0.337049 -0.791712   
 2  0.033621  0.045311  0.066122   22.528953 -1.847562 -1.229351   
 3  0.027492  0.042395  0.065891   22.590216 -0.765335 -0.875146   
 4  0.041126  0.046411  0.064277   22.644908 -0.002529 -0.582487   
 
    parkinson_vol  y_vol_next  y_up  y_high_vol  
 0       0.018471    0.028036     1           0  
 1       0.028036    0.019

## 7) Save the dataset (so the whole group can reuse it)

This saves a clean dataset file to the repository so anyone can load it later.

This is also useful if we later add sentiment features:
we can merge sentiment by Date on top of this market-only dataset.


In [8]:
# Save the final ML dataset
OUTPUT_PATH = "../data/btc_features_targets_2019_2021.csv"
dataset.to_csv(OUTPUT_PATH, index=False)

print(f"Saved dataset to {OUTPUT_PATH}")


Saved dataset to ../data/btc_features_targets_2019_2021.csv


## 8) Prepare X and y + choose the evaluation method

We split our table into:
- X = features
- y = targets

For evaluation, we use TimeSeriesSplit (walk-forward cross-validation).
This is crucial for time-series ML because random shuffling would leak future information.


In [22]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score, accuracy_score
from sklearn.linear_model import Ridge, LogisticRegression


In [ ]:

# Features
X = dataset[feature_cols]

# Targets
y_vol = dataset["y_vol_next"]        # regression
y_up = dataset["y_up"]               # classification (direction)
y_high_vol = dataset["y_high_vol"]   # classification (vol regime)

# Time-series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

print("X shape:", X.shape)
print("y_vol shape:", y_vol.shape)
print("y_up mean (class balance):", y_up.mean())
print("y_high_vol mean (class balance):", y_high_vol.mean())


X shape: (1399, 13)
y_vol shape: (1399,)
y_up mean (class balance): 0.5275196568977841
y_high_vol mean (class balance): 0.2087205146533238


## 9) Baseline model for volatility: Ridge Regression (linear + regularization)

We start with a simple, interpretable baseline.
Ridge regression is a linear model that prevents overfitting using L2 regularization.

We evaluate using time-series CV and report:
- MAE (average absolute error)
- RMSE (punishes large mistakes more)

This gives us a strong benchmark before trying more complex models.


In [ ]:
# Ridge Regression (volatility prediction)
mae_scores = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_vol.iloc[train_idx], y_vol.iloc[test_idx]
    
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    
    print(f"Fold {fold}: MAE={mae:.5f}, RMSE={rmse:.5f}")

print("\nAverage MAE:", np.mean(mae_scores))
print("Average RMSE:", np.mean(rmse_scores))


Fold 1: MAE=0.02026, RMSE=0.02426
Fold 2: MAE=0.01629, RMSE=0.02031
Fold 3: MAE=0.01365, RMSE=0.02430
Fold 4: MAE=0.01376, RMSE=0.02090
Fold 5: MAE=0.01331, RMSE=0.02209

Average MAE: 0.01545161823068104
Average RMSE: 0.022372513686732782


## 10) Baseline model for direction: Logistic Regression

Next we predict whether tomorrow’s return is positive (up/down).
Direction prediction is famously difficult in finance, so the baseline sets expectations.

We report:
- Accuracy
- F1-score

F1 is helpful when the model makes imbalanced predictions.


In [24]:
# Logistic Regression (price direction)

f1_scores = []
acc_scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_up.iloc[train_idx], y_up.iloc[test_idx]
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    
    f1_scores.append(f1)
    acc_scores.append(acc)
    
    print(f"Fold {fold}: F1={f1:.3f}, Acc={acc:.3f}")

print("\nAverage F1:", np.mean(f1_scores))
print("Average Accuracy:", np.mean(acc_scores))


Fold 1: F1=0.396, Acc=0.438
Fold 2: F1=0.653, Acc=0.485
Fold 3: F1=0.510, Acc=0.455
Fold 4: F1=0.707, Acc=0.562
Fold 5: F1=0.682, Acc=0.524

Average F1: 0.589598008050432
Average Accuracy: 0.49270386266094424


## 11) Baseline model for high-volatility regime: Logistic Regression

High-vol days are relatively rare (~20%), so accuracy is misleading.
A model that always predicts “not high-vol” could still look accurate.

So we focus on **F1-score**, which balances precision and recall for the positive class.


In [25]:
f1_scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_high_vol.iloc[train_idx], y_high_vol.iloc[test_idx]
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    f1 = f1_score(y_test, y_pred)
    f1_scores.append(f1)
    
    print(f"Fold {fold}: F1={f1:.3f}")

print("\nAverage F1 (high-vol):", np.mean(f1_scores))


Fold 1: F1=0.000
Fold 2: F1=0.300
Fold 3: F1=0.312
Fold 4: F1=0.088
Fold 5: F1=0.143

Average F1 (high-vol): 0.168718487394958


## 12) Stronger model: CatBoost for high-volatility regime

Logistic regression is linear, so it may miss nonlinear interactions.
CatBoost (tree-based) can capture:
- thresholds
- feature interactions
- nonlinear relationships

We use the same TimeSeriesSplit setup and compare F1-score to the baseline.
If CatBoost improves F1, it means nonlinear structure matters.


In [26]:
from catboost import CatBoostClassifier

f1_scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_high_vol.iloc[train_idx], y_high_vol.iloc[test_idx]
    
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="F1",
        verbose=False,
        random_seed=42
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    f1 = f1_score(y_test, y_pred)
    f1_scores.append(f1)
    
    print(f"Fold {fold}: F1={f1:.3f}")

print("\nAverage F1 (CatBoost high-vol):", np.mean(f1_scores))


Fold 1: F1=0.030
Fold 2: F1=0.434
Fold 3: F1=0.394
Fold 4: F1=0.262
Fold 5: F1=0.342

Average F1 (CatBoost high-vol): 0.2923923816546896


Compared to a linear logistic regression baseline (F1 ≈ 0.17), CatBoost significantly improves high-volatility regime detection (average F1 ≈ 0.29). This suggests that volatility regimes depend on nonlinear interactions between returns, volatility, and volume features, which linear models fail to capture.

## 13) Stronger model: CatBoost for volatility regression

Now we test whether a nonlinear model can also improve next-day volatility prediction.
We again compare against Ridge regression.

We report MAE and RMSE, using the same time-series cross-validation for fairness.


In [28]:
# can a non linear model improve volatility prediction?
from catboost import CatBoostRegressor

mae_scores = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_vol.iloc[train_idx], y_vol.iloc[test_idx]
    
    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        verbose=False,
        random_seed=42
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    
    print(f"Fold {fold}: MAE={mae:.5f}, RMSE={rmse:.5f}")

print("\nAverage MAE (CatBoost):", np.mean(mae_scores))
print("Average RMSE (CatBoost):", np.mean(rmse_scores))


Fold 1: MAE=0.01341, RMSE=0.01863
Fold 2: MAE=0.01283, RMSE=0.01839
Fold 3: MAE=0.01330, RMSE=0.02604
Fold 4: MAE=0.01399, RMSE=0.02091
Fold 5: MAE=0.01278, RMSE=0.02101

Average MAE (CatBoost): 0.013260345896118204
Average RMSE (CatBoost): 0.02099797599102692


In [31]:


# Dataset summary
print("DATASET SUMMARY")

print(f"Rows: {len(dataset)}")
print(f"Features: {X.shape[1]}")
print(f"Date range: {dataset['Date'].min().date()} → {dataset['Date'].max().date()}")
print(f"Up days (y_up mean): {y_up.mean():.3f}  (~{100*y_up.mean():.1f}% of days)")
print(f"High-vol days (y_high_vol mean): {y_high_vol.mean():.3f}  (~{100*y_high_vol.mean():.1f}% of days)")

# 2) Baseline results (fill with your numbers) 
print("\n" + "="*70)
print("BASELINE MODELS (TimeSeriesSplit CV)")
print("="*70)

ridge_mae = 0.01545161823068104
ridge_rmse = 0.022372513686732782

logreg_dir_f1 = 0.589598008050432
logreg_dir_acc = 0.49270386266094424

logreg_hv_f1 = 0.168718487394958

print(f"Volatility regression (Ridge):      MAE={ridge_mae:.5f}, RMSE={ridge_rmse:.5f}")
print(f"Direction (Logistic Regression):    F1={logreg_dir_f1:.3f}, Acc={logreg_dir_acc:.3f}")
print(f"High-vol regime (Logistic Reg.):    F1={logreg_hv_f1:.3f}")

# 3) Final model results 
print("\n" + "="*70)
print("FINAL MODELS (CatBoost, TimeSeriesSplit CV)")
print("="*70)

catboost_hv_f1 = 0.2923923816546896
catboost_mae = 0.013260345896118204
catboost_rmse = 0.02099797599102692

print(f"High-vol regime (CatBoost):         F1={catboost_hv_f1:.3f}")
print(f"Volatility regression (CatBoost):   MAE={catboost_mae:.5f}, RMSE={catboost_rmse:.5f}")

# 4) Improvements
print("\n" + "="*70)
print("IMPROVEMENT VS BASELINE")
print("="*70)

print(f"High-vol F1: {logreg_hv_f1:.3f} → {catboost_hv_f1:.3f}  (Δ = {catboost_hv_f1 - logreg_hv_f1:+.3f})")
print(f"Vol MAE:     {ridge_mae:.5f} → {catboost_mae:.5f}  (Δ = {catboost_mae - ridge_mae:+.5f})")
print(f"Vol RMSE:    {ridge_rmse:.5f} → {catboost_rmse:.5f}  (Δ = {catboost_rmse - ridge_rmse:+.5f})")


DATASET SUMMARY
Rows: 1399
Features: 13
Date range: 2018-03-02 → 2021-12-29
Up days (y_up mean): 0.528  (~52.8% of days)
High-vol days (y_high_vol mean): 0.209  (~20.9% of days)

BASELINE MODELS (TimeSeriesSplit CV)
Volatility regression (Ridge):      MAE=0.01545, RMSE=0.02237
Direction (Logistic Regression):    F1=0.590, Acc=0.493
High-vol regime (Logistic Reg.):    F1=0.169

FINAL MODELS (CatBoost, TimeSeriesSplit CV)
High-vol regime (CatBoost):         F1=0.292
Volatility regression (CatBoost):   MAE=0.01326, RMSE=0.02100

IMPROVEMENT VS BASELINE
High-vol F1: 0.169 → 0.292  (Δ = +0.124)
Vol MAE:     0.01545 → 0.01326  (Δ = -0.00219)
Vol RMSE:    0.02237 → 0.02100  (Δ = -0.00137)
